In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ============================================================
# 🚀 HINGLISH JOINT NLP MODEL — Google Colab Notebook v2
# Intent Classification + Sentiment Analysis + Slot Filling
# Using XLM-RoBERTa (Base / Large)
# ============================================================
# ✅ Fixed: tab-separated CSV loading
# ✅ Fixed: NaN values in cs_parse column
# ✅ Fixed: robust BIO tagging with type guards
# ✅ No torchcrf — uses simple softmax for slots
# ✅ Runs directly on Google Colab (T4 GPU recommended)
# ============================================================


# ─────────────────────────────────────────────
# SECTION 1: ENVIRONMENT SETUP
# ─────────────────────────────────────────────

# Run this cell FIRST in Colab before anything else:
!pip install transformers datasets seqeval scikit-learn torch -q

import os
import re
import json
import random
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from seqeval.metrics import f1_score, classification_report as seq_report

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# ─── Reproducibility ───
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

# ─── Device ───
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

# ─── Model choice ───
# Change to "xlm-roberta-large" for large model (needs more GPU RAM)
MODEL_NAME = "xlm-roberta-large"
print(f"✅ Model: {MODEL_NAME}")


# ─────────────────────────────────────────────
# SECTION 2: DATA LOADING
# ─────────────────────────────────────────────

# ── FIX: Your CSV is TAB-separated so we use sep="\t" ──
df = pd.read_csv("/content/hinglish_support_dataset_expanded_final (2).tsv", sep="\t")

# Strip any accidental whitespace from column names
df.columns = df.columns.str.strip()

print(f"✅ Dataset loaded: {len(df)} rows")
print(f"   Columns: {df.columns.tolist()}")

# ── FIX: Fill NaN values BEFORE any processing ──
df["cs_query"]  = df["cs_query"].fillna("").astype(str)
df["cs_parse"]  = df["cs_parse"].fillna("").astype(str)
df["intent"]    = df["intent"].fillna("UNKNOWN").astype(str)
df["sentiment"] = df["sentiment"].fillna("NEUTRAL").astype(str)

# Drop rows where cs_query is empty (nothing to train on)
df = df[df["cs_query"].str.strip() != ""].reset_index(drop=True)
print(f"   Rows after dropping empty queries: {len(df)}")

print(df[["cs_query", "intent", "sentiment"]].head())
print("\nIntent distribution:")
print(df["intent"].value_counts())
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())


# ─────────────────────────────────────────────
# SECTION 3: TEXT PREPROCESSING
# ─────────────────────────────────────────────

# Hinglish normalization map
HINGLISH_NORMALIZE = {
    r"\bnhi\b":    "nahi",
    r"\bnai\b":    "nahi",
    r"\bkaroon\b": "karu",
    r"\bkarna\b":  "karo",
    r"\bho gya\b": "ho gaya",
    r"\bkr\b":     "kar",
    r"\bpls\b":    "please",
    r"\bplz\b":    "please",
    r"\bthx\b":    "thanks",
    r"\bty\b":     "thanks",
    r"\brupee\b":  "rupaye",
    r"\brs\b":     "rupaye",
}

def normalize_hinglish(text):
    # Guard: ensure text is a string
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    for pattern, replacement in HINGLISH_NORMALIZE.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"\s+", " ", text)
    return text

df["cs_query_clean"] = df["cs_query"].apply(normalize_hinglish)

# Drop any rows that became empty after normalization
df = df[df["cs_query_clean"].str.strip() != ""].reset_index(drop=True)
print(f"✅ Text normalized — {len(df)} rows remaining")
print(df[["cs_query", "cs_query_clean"]].head(3))


# ─────────────────────────────────────────────
# SECTION 4: BIO TAG CONVERSION
# ─────────────────────────────────────────────

def parse_to_bio(query, parse_string):
    """
    Convert parse string like:
      [IN:COMPLAINT_DELIVERY [SL:ORDER_ID #78342 ] abhi tak nahi aaya ]
    Into BIO tags per token.

    ── FIX: Full type guard — handles NaN / float / None safely ──
    Returns: list of (token, bio_tag) tuples
    """
    # Type guard
    if not isinstance(query, str) or not isinstance(parse_string, str):
        tokens = str(query).lower().split() if isinstance(query, str) else []
        return [(t, "O") for t in tokens]

    # Extra safety: empty strings
    query = query.strip()
    parse_string = parse_string.strip()
    if not query:
        return []

    tokens = query.lower().split()
    bio_tags = ["O"] * len(tokens)

    # Extract all [SL:TYPE content ] blocks
    slot_pattern = re.compile(r"\[SL:(\w+)\s+(.*?)\s*\]")
    slots = slot_pattern.findall(parse_string)

    for slot_type, slot_content in slots:
        slot_content = slot_content.strip()
        if not slot_content:
            continue
        slot_tokens = slot_content.lower().split()
        n = len(slot_tokens)
        if n == 0:
            continue

        # Sliding window match
        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == slot_tokens:
                bio_tags[i] = f"B-{slot_type}"
                for j in range(1, n):
                    bio_tags[i+j] = f"I-{slot_type}"
                break

    return list(zip(tokens, bio_tags))


# ── Fill NaN one more time just before apply (belt + suspenders) ──
df["cs_parse"]      = df["cs_parse"].fillna("").astype(str)
df["cs_query_clean"] = df["cs_query_clean"].fillna("").astype(str)

# Apply BIO tagging to all rows
df["bio_tags"] = df.apply(
    lambda row: parse_to_bio(row["cs_query_clean"], row["cs_parse"]),
    axis=1
)

# Drop rows that produced empty bio_tags
df = df[df["bio_tags"].apply(lambda x: len(x) > 0)].reset_index(drop=True)

print(f"✅ BIO tags created — {len(df)} rows")
print(f"   Rows with no slots (all O): "
      f"{df['bio_tags'].apply(lambda x: all(t=='O' for _,t in x)).sum()}")

# Show a sample
print("\nBIO tagging example (row 0):")
for token, tag in df.iloc[0]["bio_tags"]:
    print(f"  {token:25s} → {tag}")


# ─────────────────────────────────────────────
# SECTION 5: LABEL ENCODING
# ─────────────────────────────────────────────

# Intent encoding
intent_encoder = LabelEncoder()
df["intent_id"] = intent_encoder.fit_transform(df["intent"])
intent_labels = list(intent_encoder.classes_)
num_intents = len(intent_labels)
print(f"\n✅ Intents ({num_intents}): {intent_labels}")

# Sentiment encoding
sentiment_encoder = LabelEncoder()
df["sentiment_id"] = sentiment_encoder.fit_transform(df["sentiment"])
sentiment_labels = list(sentiment_encoder.classes_)
num_sentiments = len(sentiment_labels)
print(f"✅ Sentiments ({num_sentiments}): {sentiment_labels}")

# Slot label mapping — collect all BIO tags across all rows
all_slot_tags = set()
for bio_list in df["bio_tags"]:
    for _, tag in bio_list:
        all_slot_tags.add(tag)

slot_labels = sorted(list(all_slot_tags))
# Ensure O is always index 0
if "O" in slot_labels:
    slot_labels.remove("O")
slot_labels = ["O"] + slot_labels

slot2id = {tag: idx for idx, tag in enumerate(slot_labels)}
id2slot  = {idx: tag for tag, idx in slot2id.items()}
num_slots = len(slot_labels)
print(f"✅ Slot labels ({num_slots}): {slot_labels[:10]}{'...' if num_slots>10 else ''}")


# ─────────────────────────────────────────────
# SECTION 6: TOKENIZATION & ALIGNMENT
# ─────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 128
print(f"\n✅ Tokenizer loaded: {MODEL_NAME}")

def tokenize_and_align_labels(query_tokens, bio_tags_list, max_len=MAX_LEN):
    """
    Tokenize word list and align slot labels to subword tokens.
    First subword  → real label
    Other subwords → -100  (ignored in loss calculation)
    Special tokens → -100
    """
    encoding = tokenizer(
        query_tokens,
        is_split_into_words=True,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    word_ids = encoding.word_ids(batch_index=0)
    aligned_labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)                          # [CLS]/[SEP]/[PAD]
        elif word_id != prev_word_id:
            tag = bio_tags_list[word_id] if word_id < len(bio_tags_list) else "O"
            aligned_labels.append(slot2id.get(tag, slot2id["O"]))  # first subword
        else:
            aligned_labels.append(-100)                          # continuation subword
        prev_word_id = word_id

    return {
        "input_ids":      encoding["input_ids"].squeeze(),
        "attention_mask": encoding["attention_mask"].squeeze(),
        "slot_labels":    torch.tensor(aligned_labels, dtype=torch.long)
    }

# Quick sanity check
_row = df.iloc[0]
_tokens = [t for t, _ in _row["bio_tags"]]
_tags   = [tag for _, tag in _row["bio_tags"]]
_aligned = tokenize_and_align_labels(_tokens, _tags)
print("✅ Token alignment test passed")
print(f"   input_ids shape  : {_aligned['input_ids'].shape}")
print(f"   slot_labels shape: {_aligned['slot_labels'].shape}")


# ─────────────────────────────────────────────
# SECTION 7: DATASET CLASS
# ─────────────────────────────────────────────

class HinglishDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        tokens_list = [t   for t, _ in row["bio_tags"]]
        tags_list   = [tag for _, tag in row["bio_tags"]]

        aligned = tokenize_and_align_labels(tokens_list, tags_list)

        return {
            "input_ids":       aligned["input_ids"],
            "attention_mask":  aligned["attention_mask"],
            "intent_label":    torch.tensor(row["intent_id"],    dtype=torch.long),
            "sentiment_label": torch.tensor(row["sentiment_id"], dtype=torch.long),
            "slot_labels":     aligned["slot_labels"],
        }


# ── Train / Validation split ──
# stratify only works when every class has >=2 samples
try:
    train_df, val_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df["intent"]
    )
except ValueError:
    print("⚠️  Some intent classes have <2 samples — splitting without stratify")
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"\n✅ Split → Train: {len(train_df)} | Val: {len(val_df)}")

train_dataset = HinglishDataset(train_df)
val_dataset   = HinglishDataset(val_df)

# Use batch_size=4 if you switch to xlm-roberta-large
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ DataLoaders ready (batch_size={BATCH_SIZE})")


# ─────────────────────────────────────────────
# SECTION 8: JOINT MODEL ARCHITECTURE
# ─────────────────────────────────────────────

class JointHinglishModel(nn.Module):
    """
    Shared XLM-R encoder with 3 task heads:
      1. Intent Classification   — CLS token → Linear → softmax
      2. Sentiment Analysis      — CLS token → Linear → softmax
      3. Slot Filling            — all tokens → Linear → softmax (no CRF)
    """
    def __init__(self, model_name, num_intents, num_sentiments, num_slots, dropout=0.3):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(model_name)
        hidden_size    = self.encoder.config.hidden_size

        self.dropout         = nn.Dropout(dropout)
        self.intent_head     = nn.Linear(hidden_size, num_intents)
        self.sentiment_head  = nn.Linear(hidden_size, num_sentiments)
        self.slot_head       = nn.Linear(hidden_size, num_slots)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        # [CLS] token for sentence-level tasks
        cls_output   = self.dropout(outputs.last_hidden_state[:, 0, :])   # (B, H)
        # All tokens for slot filling
        token_output = self.dropout(outputs.last_hidden_state)             # (B, S, H)

        intent_logits    = self.intent_head(cls_output)      # (B, num_intents)
        sentiment_logits = self.sentiment_head(cls_output)   # (B, num_sentiments)
        slot_logits      = self.slot_head(token_output)      # (B, S, num_slots)

        return intent_logits, sentiment_logits, slot_logits


model = JointHinglishModel(
    model_name=MODEL_NAME,
    num_intents=num_intents,
    num_sentiments=num_sentiments,
    num_slots=num_slots,
    dropout=0.3
).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✅ Model loaded on {device}")
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")


# ─────────────────────────────────────────────
# SECTION 9: LOSS FUNCTION & OPTIMIZER
# ─────────────────────────────────────────────

# ── Class weights to handle imbalanced intents / sentiments ──
intent_counts    = np.array([
    (df["intent_id"] == i).sum() for i in range(num_intents)
], dtype=float)
sentiment_counts = np.array([
    (df["sentiment_id"] == i).sum() for i in range(num_sentiments)
], dtype=float)

# Avoid divide-by-zero for any class with 0 samples
intent_counts    = np.maximum(intent_counts,    1)
sentiment_counts = np.maximum(sentiment_counts, 1)

intent_weights    = torch.tensor(1.0 / intent_counts,    dtype=torch.float).to(device)
sentiment_weights = torch.tensor(1.0 / sentiment_counts, dtype=torch.float).to(device)
intent_weights    = intent_weights    / intent_weights.sum()    * num_intents
sentiment_weights = sentiment_weights / sentiment_weights.sum() * num_sentiments

# Loss functions
intent_loss_fn    = nn.CrossEntropyLoss(weight=intent_weights)
sentiment_loss_fn = nn.CrossEntropyLoss(weight=sentiment_weights)
slot_loss_fn      = nn.CrossEntropyLoss(ignore_index=-100)   # -100 = subword / special token

# Loss weights — slots are harder so get 2× weight
ALPHA = 1.0   # intent weight
BETA  = 1.0   # sentiment weight
GAMMA = 2.0   # slot weight

def compute_joint_loss(intent_logits, sentiment_logits, slot_logits,
                        intent_labels, sentiment_labels, slot_labels):
    """
    Total Loss = α·IntentLoss + β·SentimentLoss + γ·SlotLoss
    """
    l_intent    = intent_loss_fn(intent_logits, intent_labels)
    l_sentiment = sentiment_loss_fn(sentiment_logits, sentiment_labels)

    batch, seq, n_slots = slot_logits.shape
    l_slot = slot_loss_fn(
        slot_logits.view(batch * seq, n_slots),
        slot_labels.view(batch * seq)
    )

    total = ALPHA * l_intent + BETA * l_sentiment + GAMMA * l_slot
    return total, l_intent, l_sentiment, l_slot


# ── Optimizer & scheduler ──
# Use LEARNING_RATE = 1e-5 if you switch to xlm-roberta-large
LEARNING_RATE = 1e-5
NUM_EPOCHS    = 10

optimizer    = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps  = len(train_loader) * NUM_EPOCHS
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
print(f"\n✅ Optimizer ready — LR: {LEARNING_RATE} | Epochs: {NUM_EPOCHS} | "
      f"Total steps: {total_steps}")


# ─────────────────────────────────────────────
# SECTION 10: TRAINING LOOP
# ─────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    all_intent_preds, all_intent_true = [], []
    all_sent_preds,   all_sent_true   = [], []

    for batch in loader:
        input_ids        = batch["input_ids"].to(device)
        attention_mask   = batch["attention_mask"].to(device)
        intent_labels    = batch["intent_label"].to(device)
        sentiment_labels = batch["sentiment_label"].to(device)
        slot_labels      = batch["slot_labels"].to(device)

        optimizer.zero_grad()

        intent_logits, sentiment_logits, slot_logits = model(input_ids, attention_mask)

        loss, l_i, l_s, l_sl = compute_joint_loss(
            intent_logits, sentiment_logits, slot_logits,
            intent_labels, sentiment_labels, slot_labels
        )

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        all_intent_preds.extend(intent_logits.argmax(-1).cpu().tolist())
        all_intent_true.extend(intent_labels.cpu().tolist())
        all_sent_preds.extend(sentiment_logits.argmax(-1).cpu().tolist())
        all_sent_true.extend(sentiment_labels.cpu().tolist())

    avg_loss   = total_loss / len(loader)
    intent_acc = accuracy_score(all_intent_true, all_intent_preds)
    sent_acc   = accuracy_score(all_sent_true,   all_sent_preds)
    return avg_loss, intent_acc, sent_acc


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_intent_preds, all_intent_true = [], []
    all_sent_preds,   all_sent_true   = [], []
    all_slot_preds,   all_slot_true   = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids        = batch["input_ids"].to(device)
            attention_mask   = batch["attention_mask"].to(device)
            intent_labels    = batch["intent_label"].to(device)
            sentiment_labels = batch["sentiment_label"].to(device)
            slot_labels      = batch["slot_labels"].to(device)

            intent_logits, sentiment_logits, slot_logits = model(input_ids, attention_mask)

            loss, _, _, _ = compute_joint_loss(
                intent_logits, sentiment_logits, slot_logits,
                intent_labels, sentiment_labels, slot_labels
            )
            total_loss += loss.item()

            all_intent_preds.extend(intent_logits.argmax(-1).cpu().tolist())
            all_intent_true.extend(intent_labels.cpu().tolist())
            all_sent_preds.extend(sentiment_logits.argmax(-1).cpu().tolist())
            all_sent_true.extend(sentiment_labels.cpu().tolist())

            # Slot predictions — skip -100 positions
            slot_pred_ids = slot_logits.argmax(-1).cpu().tolist()
            slot_true_ids = slot_labels.cpu().tolist()

            for pred_seq, true_seq in zip(slot_pred_ids, slot_true_ids):
                pred_tags, true_tags = [], []
                for p, t in zip(pred_seq, true_seq):
                    if t == -100:
                        continue
                    pred_tags.append(id2slot[p])
                    true_tags.append(id2slot[t])
                all_slot_preds.append(pred_tags)
                all_slot_true.append(true_tags)

    avg_loss   = total_loss / len(loader)
    intent_acc = accuracy_score(all_intent_true, all_intent_preds)
    sent_acc   = accuracy_score(all_sent_true,   all_sent_preds)

    try:
        slot_f1 = f1_score(all_slot_true, all_slot_preds)
    except Exception:
        slot_f1 = 0.0

    return avg_loss, intent_acc, sent_acc, slot_f1


# ── Training with early stopping ──
PATIENCE         = 3
best_val_loss    = float("inf")
patience_counter = 0
history          = []

print("\n🚀 Starting Training...\n")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>8} | "
      f"{'Intent Acc':>10} | {'Sent Acc':>8} | {'Slot F1':>7}")
print("─" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_intent_acc, train_sent_acc = train_epoch(
        model, train_loader, optimizer, scheduler
    )
    val_loss, val_intent_acc, val_sent_acc, val_slot_f1 = evaluate(
        model, val_loader
    )

    history.append({
        "epoch":          epoch,
        "train_loss":     train_loss,
        "val_loss":       val_loss,
        "val_intent_acc": val_intent_acc,
        "val_sent_acc":   val_sent_acc,
        "val_slot_f1":    val_slot_f1,
    })

    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>8.4f} | "
          f"{val_intent_acc:>10.4f} | {val_sent_acc:>8.4f} | {val_slot_f1:>7.4f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model1.pt")
        print(f"         ✅ Best model saved (val_loss={val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⚠️  Early stopping triggered at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs)")
            break

print("\n✅ Training complete!")


# ─────────────────────────────────────────────
# SECTION 11: LOAD BEST MODEL & FINAL EVALUATION
# ─────────────────────────────────────────────

model.load_state_dict(torch.load("best_model1.pt", map_location=device))
print("✅ Best model loaded")

val_loss, val_intent_acc, val_sent_acc, val_slot_f1 = evaluate(model, val_loader)

print(f"\n{'='*45}")
print(f"  📊 Final Validation Metrics")
print(f"{'='*45}")
print(f"  Intent Accuracy   : {val_intent_acc*100:.2f}%")
print(f"  Sentiment Accuracy: {val_sent_acc*100:.2f}%")
print(f"  Slot F1 Score     : {val_slot_f1*100:.2f}%")
print(f"{'='*45}")


# ─────────────────────────────────────────────
# SECTION 12: VISUALIZATION
# ─────────────────────────────────────────────

def plot_confusion_matrix(true_labels, pred_labels, label_names, title):
    cm = confusion_matrix(true_labels, pred_labels, labels=label_names)
    fig_size = max(6, len(label_names))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size - 1))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    fname = title.replace(" ", "_") + "1.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"✅ Saved: {fname}")

# Collect val predictions for confusion matrices
model.eval()
all_intent_preds, all_intent_true = [], []
all_sent_preds,   all_sent_true   = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        intent_logits, sentiment_logits, _ = model(input_ids, attention_mask)
        all_intent_preds.extend(intent_logits.argmax(-1).cpu().tolist())
        all_intent_true.extend(batch["intent_label"].tolist())
        all_sent_preds.extend(sentiment_logits.argmax(-1).cpu().tolist())
        all_sent_true.extend(batch["sentiment_label"].tolist())

true_intent_names = intent_encoder.inverse_transform(all_intent_true)
pred_intent_names = intent_encoder.inverse_transform(all_intent_preds)
true_sent_names   = sentiment_encoder.inverse_transform(all_sent_true)
pred_sent_names   = sentiment_encoder.inverse_transform(all_sent_preds)

plot_confusion_matrix(true_intent_names, pred_intent_names, intent_labels,
                      "Intent Confusion Matrix")
plot_confusion_matrix(true_sent_names,   pred_sent_names,   sentiment_labels,
                      "Sentiment Confusion Matrix")

# ── Training curves ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_list = [h["epoch"] for h in history]

axes[0].plot(epochs_list, [h["train_loss"]     for h in history], label="Train", marker="o")
axes[0].plot(epochs_list, [h["val_loss"]       for h in history], label="Val",   marker="o")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(epochs_list, [h["val_intent_acc"] for h in history], color="green", marker="o")
axes[1].set_title("Val Intent Accuracy"); axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)

axes[2].plot(epochs_list, [h["val_slot_f1"]    for h in history], color="orange", marker="o")
axes[2].set_title("Val Slot F1"); axes[2].set_xlabel("Epoch")
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("training_curves1.png", dpi=150)
plt.show()
print("✅ Training curves saved")


# ─────────────────────────────────────────────
# SECTION 13: INFERENCE FUNCTION
# ─────────────────────────────────────────────

def predict(text, model=model):
    """
    Input : Hinglish sentence (string)
    Output: dict with intent, sentiment, and extracted slots
    """
    model.eval()

    text_clean = normalize_hinglish(text)
    tokens     = text_clean.split()

    if not tokens:
        return {"input": text, "intent": "UNKNOWN", "sentiment": "UNKNOWN", "slots": {}}

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    input_ids      = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    word_ids       = encoding.word_ids(batch_index=0)

    with torch.no_grad():
        intent_logits, sentiment_logits, slot_logits = model(input_ids, attention_mask)

    # ── Decode intent & sentiment ──
    intent_id    = intent_logits.argmax(-1).item()
    sentiment_id = sentiment_logits.argmax(-1).item()
    pred_intent    = intent_encoder.inverse_transform([intent_id])[0]
    pred_sentiment = sentiment_encoder.inverse_transform([sentiment_id])[0]

    # ── Decode slots ──
    slot_pred_ids  = slot_logits.argmax(-1).squeeze().tolist()
    slot_predictions = {}
    prev_word_id   = None
    current_slot   = None
    current_values = []

    for idx, word_id in enumerate(word_ids):
        if word_id is None or word_id == prev_word_id:
            continue
        tag  = id2slot[slot_pred_ids[idx]]
        word = tokens[word_id] if word_id < len(tokens) else ""

        if tag.startswith("B-"):
            if current_slot:
                slot_predictions[current_slot] = " ".join(current_values)
            current_slot   = tag[2:]
            current_values = [word]
        elif tag.startswith("I-") and current_slot:
            current_values.append(word)
        else:
            if current_slot:
                slot_predictions[current_slot] = " ".join(current_values)
                current_slot   = None
                current_values = []

        prev_word_id = word_id

    if current_slot:
        slot_predictions[current_slot] = " ".join(current_values)

    return {
        "input":     text,
        "intent":    pred_intent,
        "sentiment": pred_sentiment,
        "slots":     slot_predictions
    }


# ── Test inference ──
test_sentences = [
    "Mera order abhi tak nahi aaya 5 din ho gaye",
    "200 rupaye zyada charge kar liye please refund karo",
    "payment fail ho gayi lekin paise kat gaye",
    "PhonePe se UPI payment nahi ho rahi kya karu",
    "complaint 2 hafte se pending hai koi sun nahi raha",
]

print("\n🔍 Inference Examples:\n")
for sentence in test_sentences:
    result = predict(sentence)
    print(f"Input    : {result['input']}")
    print(f"Intent   : {result['intent']}")
    print(f"Sentiment: {result['sentiment']}")
    print(f"Slots    : {result['slots']}")
    print("─" * 60)


# ─────────────────────────────────────────────
# SECTION 14: SAVE MODEL
# ─────────────────────────────────────────────

# Save model weights
torch.save(model.state_dict(), "hinglish_joint_model1.pt")

# Save all label mappings needed for inference
metadata = {
    "intent_labels":    intent_labels,
    "sentiment_labels": sentiment_labels,
    "slot_labels":      slot_labels,
    "slot2id":          slot2id,
    "id2slot":          {str(k): v for k, v in id2slot.items()},  # JSON needs str keys
    "model_name":       MODEL_NAME,
    "max_len":          MAX_LEN,
}
with open("model_metadata2.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Model saved  : hinglish_joint_model1.pt")
print("✅ Metadata saved: model_metadata2.json")

from google.colab import drive
drive.mount('/content/drive')

!cp hinglish_joint_model1.pt /content/drive/MyDrive/XLmLarge
!cp model_metadata2.json /content/drive/MyDrive/XLmLarge
!cp best_model1.pt /content/drive/MyDrive/XLmLarge

# ── Download from Colab ──
# !zip -r hinglish_model.zip hinglish_joint_model.pt model_metadata.json best_model.pt
# from google.colab import files
# files.download("hinglish_model.zip")


# ─────────────────────────────────────────────
# SECTION 15: SWITCH TO XLM-R LARGE (OPTIONAL)
# ─────────────────────────────────────────────
# To switch to xlm-roberta-large, change these 3 lines at the top and re-run:
#
#   MODEL_NAME    = "xlm-roberta-large"
#   BATCH_SIZE    = 4        # large needs more GPU RAM
#   LEARNING_RATE = 1e-5     # slightly smaller LR for large
#
# xlm-roberta-large needs ~10GB GPU RAM.
# Free Colab T4 (15GB) can handle it with batch_size=4.
print("\n✅ See Section 15 comments to switch to XLM-R Large")


# ─────────────────────────────────────────────
# SECTION 16: ADVANCED — ADD CRF (OPTIONAL)
# ─────────────────────────────────────────────
# CRF improves slot F1 by ~2-4% by enforcing valid BIO transitions.
# Only add AFTER the base softmax model is working correctly.
#
# Install:  !pip install torchcrf -q
#
# Then replace the slot_head forward pass with:
#
# from torchcrf import CRF
# self.crf = CRF(num_slots, batch_first=True)
#
# In forward():
#   slot_emissions = self.slot_head(token_output)
#   if slot_labels is not None:
#       mask = (slot_labels != -100)
#       labels_crf = slot_labels.clone(); labels_crf[~mask] = 0
#       crf_loss = -self.crf(slot_emissions, labels_crf, mask=mask, reduction="mean")
#       return intent_logits, sentiment_logits, crf_loss
#   else:
#       return intent_logits, sentiment_logits, self.crf.decode(slot_emissions)
print("✅ See Section 16 comments for optional CRF upgrade")

In [ ]:
# !zip -r project_files.zip . -x "sample_data/*"
# from google.colab import files
# files.download("project_files.zip")

In [ ]:
# ============================================================
# 🧪 HINGLISH JOINT MODEL — INTERACTIVE TESTER
# Run this AFTER your training notebook has completed
# All variables (model, tokenizer, encoders) must be in memory
# ============================================================


# ─────────────────────────────────────────────
# OPTION A: TEST A SINGLE SENTENCE
# ─────────────────────────────────────────────

def test_single(text):
    result = predict(text)
    print(f"\n{'='*60}")
    print(f"  INPUT     : {result['input']}")
    print(f"  INTENT    : {result['intent']}")
    print(f"  SENTIMENT : {result['sentiment']}")
    if result['slots']:
        print(f"  SLOTS     :")
        for slot_name, slot_value in result['slots'].items():
            print(f"              {slot_name:25s} → {slot_value}")
    else:
        print(f"  SLOTS     : (none detected)")
    print(f"{'='*60}")

# ── Try it ──
test_single("Mera order #45231 abhi tak nahi aaya")


# ─────────────────────────────────────────────
# OPTION B: TEST MULTIPLE SENTENCES AT ONCE
# Just add / remove sentences from this list
# ─────────────────────────────────────────────

my_test_sentences = [
    # Delivery complaints
    "Mera order 5 din se deliver nahi hua koi batao",
    "Order #78342 7 din ho gaye abhi tak nahi aaya",
    "Package deliver nahi hua aur tracking update bhi nahi hai",

    # Payment issues
    "Payment 799 rupaye cut ho gayi but order show nahi ho raha",
    "PhonePe se UPI payment fail ho gayi lekin paise kat gaye",
    "Aadhaar se payment nahi ho pa rahi kya karu",

    # Refund requests
    "Refund kab aayega 2 weeks ho gaye abhi tak nahi aaya",
    "Mujhe refund chahiye order cancel kar diya tha",
    "200 rupaye zyada charge kar liye please refund karo",

    # Product complaints
    "Earphones sirf 2 din mein kharab ho gaye bilkul bakwaas quality",
    "Jo product mila wo description se bilkul alag hai",
    "Item damaged condition mein aaya box bhi tuta hua tha",

    # Escalation
    "Complaint 2 hafte se pending hai koi sun nahi raha escalate karo",
    "Manager se baat karni hai yaar koi kaam nahi kar raha",

    # Positive feedback
    "Delivery ekdum fast thi thank you bahut acha experience raha",
    "Product quality bahut acha hai very happy with purchase",

    # Cancel requests
    "Order cancel karna hai abhi place kiya tha 5 minute pehle",
    "Prepaid order cancel karke refund milega kya",

    # Mixed / tricky
    "Bhai 3 baar complaint ki lekin koi response nahi aaya kya ho raha hai",
    "App crash ho rahi hai payment ke time pe",
]

def test_batch(sentences):
    print(f"\n{'='*60}")
    print(f"  🧪 BATCH TEST — {len(sentences)} sentences")
    print(f"{'='*60}\n")

    results = []
    for i, sentence in enumerate(sentences, 1):
        result = predict(sentence)
        results.append(result)

        slots_str = ", ".join(
            [f"{k}={v}" for k, v in result['slots'].items()]
        ) if result['slots'] else "—"

        print(f"[{i:02d}] {sentence[:55]:<55}")
        print(f"      Intent    : {result['intent']}")
        print(f"      Sentiment : {result['sentiment']}")
        print(f"      Slots     : {slots_str}")
        print()

    return results

results = test_batch(my_test_sentences)


# ─────────────────────────────────────────────
# OPTION C: RESULTS TABLE (clean view)
# ─────────────────────────────────────────────

def show_results_table(results):
    rows = []
    for r in results:
        rows.append({
            "Input"     : r["input"][:50] + ("..." if len(r["input"]) > 50 else ""),
            "Intent"    : r["intent"],
            "Sentiment" : r["sentiment"],
            "Slots"     : str(r["slots"]) if r["slots"] else "—"
        })
    df_results = pd.DataFrame(rows)
    print("\n📊 Results Table:")
    print(df_results.to_string(index=False))
    return df_results

df_results = show_results_table(results)


# ─────────────────────────────────────────────
# OPTION D: INTERACTIVE LOOP
# Type sentences one by one — type 'exit' to stop
# ─────────────────────────────────────────────

def interactive_test():
    print("\n" + "="*60)
    print("  💬 INTERACTIVE MODE")
    print("  Type any Hinglish sentence and press Enter")
    print("  Type 'exit' to stop")
    print("="*60 + "\n")

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Exiting interactive mode")
            break

        if not user_input:
            continue
        if user_input.lower() in ("exit", "quit", "q", "bye"):
            print("👋 Exiting interactive mode")
            break

        test_single(user_input)

# Uncomment the line below to start interactive mode:
# interactive_test()


# ─────────────────────────────────────────────
# OPTION E: CONFIDENCE SCORES
# See probability scores for each prediction
# ─────────────────────────────────────────────

import torch.nn.functional as F

def predict_with_confidence(text):
    model.eval()
    text_clean = normalize_hinglish(text)
    tokens     = text_clean.split()
    if not tokens:
        return None

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        intent_logits, sentiment_logits, slot_logits = model(
            encoding["input_ids"].to(device),
            encoding["attention_mask"].to(device)
        )

    # Probabilities
    intent_probs    = F.softmax(intent_logits, dim=-1).squeeze().cpu()
    sentiment_probs = F.softmax(sentiment_logits, dim=-1).squeeze().cpu()

    # Top 3 intents
    top3_intent_vals, top3_intent_idx = intent_probs.topk(3)
    top3_sent_vals,   top3_sent_idx   = sentiment_probs.topk(min(3, num_sentiments))

    print(f"\n{'='*60}")
    print(f"  INPUT : {text}")
    print(f"{'─'*60}")
    print(f"  TOP INTENTS:")
    for val, idx in zip(top3_intent_vals, top3_intent_idx):
        label = intent_labels[idx.item()]
        bar   = "█" * int(val.item() * 20)
        print(f"    {label:30s} {val.item()*100:5.1f}%  {bar}")

    print(f"\n  SENTIMENT:")
    for val, idx in zip(top3_sent_vals, top3_sent_idx):
        label = sentiment_labels[idx.item()]
        bar   = "█" * int(val.item() * 20)
        print(f"    {label:15s} {val.item()*100:5.1f}%  {bar}")
    print(f"{'='*60}")


# ── Test confidence scores ──
predict_with_confidence("Mera order nahi aaya 5 din ho gaye")
predict_with_confidence("Payment fail ho gayi lekin paise kat gaye")
predict_with_confidence("Delivery bahut fast thi thank you")


# ─────────────────────────────────────────────
# OPTION F: EXPORT RESULTS TO CSV
# ─────────────────────────────────────────────

def export_results(sentences, filename="test_results.csv"):
    rows = []
    for sentence in sentences:
        r = predict(sentence)
        rows.append({
            "input"    : r["input"],
            "intent"   : r["intent"],
            "sentiment": r["sentiment"],
            "slots"    : json.dumps(r["slots"])
        })
    df_export = pd.DataFrame(rows)
    df_export.to_csv(filename, index=False)
    print(f"✅ Results saved to {filename}")
    return df_export

# Uncomment to export:
# export_results(my_test_sentences, "my_test_results.csv")
# from google.colab import files
# files.download("my_test_results.csv")

In [ ]:
# ============================================================
# 💬 HINGLISH MODEL — TYPE & PREDICT
# Paste this in a new cell AFTER your training is complete
# ============================================================

def predict_and_show(text):
    model.eval()
    text_clean = normalize_hinglish(text)
    tokens     = text_clean.split()

    if not tokens:
        print("⚠️  Please enter a valid sentence.")
        return

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        intent_logits, sentiment_logits, slot_logits = model(
            encoding["input_ids"].to(device),
            encoding["attention_mask"].to(device)
        )

    import torch.nn.functional as F

    # ── Intent (top 3 with confidence) ──
    intent_probs        = F.softmax(intent_logits, dim=-1).squeeze().cpu()
    top3_vals, top3_idx = intent_probs.topk(3)
    pred_intent         = intent_labels[top3_idx[0].item()]

    # ── Sentiment ──
    sentiment_probs  = F.softmax(sentiment_logits, dim=-1).squeeze().cpu()
    pred_sent_idx    = sentiment_probs.argmax().item()
    pred_sentiment   = sentiment_labels[pred_sent_idx]
    sent_conf        = sentiment_probs[pred_sent_idx].item()

    # ── Slots ──
    slot_pred_ids  = slot_logits.argmax(-1).squeeze().tolist()
    word_ids       = encoding.word_ids(batch_index=0)
    slot_results   = {}
    prev_word_id   = None
    current_slot   = None
    current_values = []

    for idx, word_id in enumerate(word_ids):
        if word_id is None or word_id == prev_word_id:
            continue
        tag  = id2slot[slot_pred_ids[idx]]
        word = tokens[word_id] if word_id < len(tokens) else ""

        if tag.startswith("B-"):
            if current_slot:
                slot_results[current_slot] = " ".join(current_values)
            current_slot   = tag[2:]
            current_values = [word]
        elif tag.startswith("I-") and current_slot:
            current_values.append(word)
        else:
            if current_slot:
                slot_results[current_slot] = " ".join(current_values)
                current_slot   = None
                current_values = []
        prev_word_id = word_id

    if current_slot:
        slot_results[current_slot] = " ".join(current_values)

    # ── Print output ──
    print("\n" + "━"*55)
    print(f"  📝 Input     : {text}")
    print("━"*55)
    print(f"  🎯 Intent    : {pred_intent}  ({top3_vals[0].item()*100:.1f}%)")
    print(f"     2nd guess : {intent_labels[top3_idx[1].item()]}  ({top3_vals[1].item()*100:.1f}%)")
    print(f"     3rd guess : {intent_labels[top3_idx[2].item()]}  ({top3_vals[2].item()*100:.1f}%)")
    print(f"  💬 Sentiment : {pred_sentiment}  ({sent_conf*100:.1f}%)")
    if slot_results:
        print(f"  🏷️  Slots     :")
        for k, v in slot_results.items():
            print(f"               {k} = {v}")
    else:
        print(f"  🏷️  Slots     : (none detected)")
    print("━"*55 + "\n")


# ── INTERACTIVE LOOP — just type your sentence ──
print("💬 Hinglish Model Tester")
print("   Type any sentence and press Enter")
print("   Type 'exit' to stop\n")

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Done")
        break

    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit", "q"):
        print("👋 Done")
        break

    predict_and_show(user_input)

In [ ]:
# ============================================================
# 💬 HINGLISH MODEL TESTER — COMPLETE SELF-CONTAINED
# This cell reloads everything and lets you type sentences
# Run this cell ALONE even after closing/reopening Colab
# ============================================================

# ─────────────────────────────────────────────
# STEP 1: IMPORTS
# ─────────────────────────────────────────────

import os
import re
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")


# ─────────────────────────────────────────────
# STEP 2: LOAD METADATA (saved during training)
# ─────────────────────────────────────────────

with open("model_metadata2.json", "r") as f:
    metadata = json.load(f)

MODEL_NAME      = metadata["model_name"]
MAX_LEN         = metadata["max_len"]
intent_labels   = metadata["intent_labels"]
sentiment_labels= metadata["sentiment_labels"]
slot_labels     = metadata["slot_labels"]
slot2id         = metadata["slot2id"]
id2slot         = {int(k): v for k, v in metadata["id2slot"].items()}

num_intents     = len(intent_labels)
num_sentiments  = len(sentiment_labels)
num_slots       = len(slot_labels)

# Rebuild label encoders from metadata
intent_encoder = LabelEncoder()
intent_encoder.classes_ = np.array(intent_labels)

sentiment_encoder = LabelEncoder()
sentiment_encoder.classes_ = np.array(sentiment_labels)

print(f"✅ Metadata loaded")
print(f"   Model        : {MODEL_NAME}")
print(f"   Intents      : {num_intents}")
print(f"   Sentiments   : {num_sentiments}")
print(f"   Slot labels  : {num_slots}")


# ─────────────────────────────────────────────
# STEP 3: LOAD TOKENIZER
# ─────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded")


# ─────────────────────────────────────────────
# STEP 4: REBUILD MODEL ARCHITECTURE
# (must match exactly what was used in training)
# ─────────────────────────────────────────────

class JointHinglishModel(nn.Module):
    def __init__(self, model_name, num_intents, num_sentiments, num_slots, dropout=0.3):
        super().__init__()
        self.encoder        = AutoModel.from_pretrained(model_name)
        hidden_size         = self.encoder.config.hidden_size
        self.dropout        = nn.Dropout(dropout)
        self.intent_head    = nn.Linear(hidden_size, num_intents)
        self.sentiment_head = nn.Linear(hidden_size, num_sentiments)
        self.slot_head      = nn.Linear(hidden_size, num_slots)

    def forward(self, input_ids, attention_mask):
        outputs      = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output   = self.dropout(outputs.last_hidden_state[:, 0, :])
        token_output = self.dropout(outputs.last_hidden_state)
        return (
            self.intent_head(cls_output),
            self.sentiment_head(cls_output),
            self.slot_head(token_output)
        )

model = JointHinglishModel(
    model_name=MODEL_NAME,
    num_intents=num_intents,
    num_sentiments=num_sentiments,
    num_slots=num_slots,
    dropout=0.3
).to(device)

print(f"✅ Model architecture built")


# ─────────────────────────────────────────────
# STEP 5: LOAD SAVED WEIGHTS
# ─────────────────────────────────────────────

model.load_state_dict(torch.load("best_model1.pt", map_location=device))
model.eval()
print(f"✅ Weights loaded from best_model1.pt")


# ─────────────────────────────────────────────
# STEP 6: NORMALIZATION FUNCTION
# ─────────────────────────────────────────────

HINGLISH_NORMALIZE = {
    r"\bnhi\b":    "nahi",
    r"\bnai\b":    "nahi",
    r"\bkaroon\b": "karu",
    r"\bkarna\b":  "karo",
    r"\bho gya\b": "ho gaya",
    r"\bkr\b":     "kar",
    r"\bpls\b":    "please",
    r"\bplz\b":    "please",
    r"\bthx\b":    "thanks",
    r"\bty\b":     "thanks",
    r"\brupee\b":  "rupaye",
    r"\brs\b":     "rupaye",
}

def normalize_hinglish(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    for pattern, replacement in HINGLISH_NORMALIZE.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"\s+", " ", text)
    return text


# ─────────────────────────────────────────────
# STEP 7: PREDICT FUNCTION
# ─────────────────────────────────────────────

def predict_and_show(text):
    model.eval()
    text_clean = normalize_hinglish(text)
    tokens     = text_clean.split()

    if not tokens:
        print("⚠️  Please enter a valid sentence.")
        return

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        intent_logits, sentiment_logits, slot_logits = model(
            encoding["input_ids"].to(device),
            encoding["attention_mask"].to(device)
        )

    # ── Intent — top 3 with confidence ──
    intent_probs        = F.softmax(intent_logits, dim=-1).squeeze().cpu()
    top3_vals, top3_idx = intent_probs.topk(3)
    pred_intent         = intent_labels[top3_idx[0].item()]

    # ── Sentiment ──
    sentiment_probs = F.softmax(sentiment_logits, dim=-1).squeeze().cpu()
    pred_sent_idx   = sentiment_probs.argmax().item()
    pred_sentiment  = sentiment_labels[pred_sent_idx]
    sent_conf       = sentiment_probs[pred_sent_idx].item()

    # ── Slots ──
    slot_pred_ids  = slot_logits.argmax(-1).squeeze().tolist()
    word_ids       = encoding.word_ids(batch_index=0)
    slot_results   = {}
    prev_word_id   = None
    current_slot   = None
    current_values = []

    for idx, word_id in enumerate(word_ids):
        if word_id is None or word_id == prev_word_id:
            continue
        tag  = id2slot[slot_pred_ids[idx]]
        word = tokens[word_id] if word_id < len(tokens) else ""

        if tag.startswith("B-"):
            if current_slot:
                slot_results[current_slot] = " ".join(current_values)
            current_slot   = tag[2:]
            current_values = [word]
        elif tag.startswith("I-") and current_slot:
            current_values.append(word)
        else:
            if current_slot:
                slot_results[current_slot] = " ".join(current_values)
                current_slot   = None
                current_values = []
        prev_word_id = word_id

    if current_slot:
        slot_results[current_slot] = " ".join(current_values)

    # ── Print result ──
    print("\n" + "━"*55)
    print(f"  📝 Input     : {text}")
    print("━"*55)
    print(f"  🎯 Intent    : {pred_intent}  ({top3_vals[0].item()*100:.1f}%)")
    print(f"     2nd guess : {intent_labels[top3_idx[1].item()]}  ({top3_vals[1].item()*100:.1f}%)")
    print(f"     3rd guess : {intent_labels[top3_idx[2].item()]}  ({top3_vals[2].item()*100:.1f}%)")
    print(f"  💬 Sentiment : {pred_sentiment}  ({sent_conf*100:.1f}%)")
    if slot_results:
        print(f"  🏷️  Slots     :")
        for k, v in slot_results.items():
            print(f"               {k} = {v}")
    else:
        print(f"  🏷️  Slots     : (none detected)")
    print("━"*55 + "\n")


# ─────────────────────────────────────────────
# STEP 8: START TYPING — interactive loop
# ─────────────────────────────────────────────

print("\n" + "="*55)
print("  💬 HINGLISH MODEL — TYPE YOUR SENTENCE")
print("  Type any Hinglish sentence and press Enter")
print("  Type 'exit' to stop")
print("="*55 + "\n")

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Done")
        break

    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit", "q"):
        print("👋 Done")
        break

    predict_and_show(user_input)